In [5]:
from pathlib import Path

import pandas as pd
import triplets

### Task 3.1.
First we import the  provided EQ profile and asses the total production capacity of the generators in the model.

In [6]:
model_path = Path.cwd() / "20210325T1530Z_1D_NL_EQ_001 3.xml"
triples = triplets.parse(str(model_path), return_type="pandas")

# A triplet row uses ID as the subject, KEY as the CIM property,
# and VALUE as the property value.
generator_ids = set(
    triples.loc[
        triples["VALUE"].eq("GeneratingUnit"),
        "ID",
    ]
)
generator_properties = triples[triples["ID"].isin(generator_ids)]

def property_values(property_name):
    values = generator_properties[generator_properties["KEY"].eq(property_name)]
    return values.set_index("ID")["VALUE"]

names = property_values("IdentifiedObject.name")
mrids = property_values("IdentifiedObject.mRID")
minimum_power = property_values("GeneratingUnit.minOperatingP").astype(float)
nominal_power = property_values("GeneratingUnit.nominalP").astype(float)
maximum_power = property_values("GeneratingUnit.maxOperatingP").astype(float)

generators = pd.DataFrame(
    {
        "generator": names,
        "mRID": mrids,
        "minimum_operating_power_MW": minimum_power,
        "nominal_power_MW": nominal_power,
        "maximum_operating_power_MW": maximum_power,
    }
).sort_values("generator").reset_index(drop=True)

display(generators)

maximum_capacity_mw = generators["maximum_operating_power_MW"].sum()
nominal_capacity_mw = generators["nominal_power_MW"].sum()
minimum_capacity_mw = generators["minimum_operating_power_MW"].sum()

print(f"Total maximum production capacity: {maximum_capacity_mw:.0f} MW")
print(f"Total nominal production capacity: {nominal_capacity_mw:.0f} MW")
print(f"Total minimum operating power: {minimum_capacity_mw:.0f} MW")

,generator,mRID,minimum_operating_power_MW,nominal_power_MW,maximum_operating_power_MW
0,Gen-12908,b850063d-eae7-4675-bc98-4642d3076783,130.0,225.0,250.0
1,Gen-12910,ca80ee09-3bed-4884-bc28-6dc89d067289,130.0,225.0,250.0
2,Gen-12923,049438a6-780a-44fe-a788-ebe385d98e25,300.0,990.0,1000.0


Total maximum production capacity: 1500 MW
Total nominal production capacity: 1440 MW
Total minimum operating power: 560 MW


### Task 3.2.
Now we asses the nominal voltages of the windins of the transformer NL_TR2_2 (ID:_2184f365-8cd5-4b5d-8a28-9d68603bb6a4)

In [8]:
target_transformer_id = "2184f365-8cd5-4b5d-8a28-9d68603bb6a4"

def normalize_id(value):
    return str(value).lstrip("#_")

# Find PowerTransformerEnd objects that explicitly reference NL_TR2_2.
end_type_ids = set(
    triples.loc[
        triples["VALUE"].eq("PowerTransformerEnd"),
        "ID",
    ]
)
end_properties = triples[triples["ID"].isin(end_type_ids)].copy()
linked_end_ids = set(
    end_properties.loc[
        end_properties["KEY"].eq("PowerTransformerEnd.PowerTransformer")
        & end_properties["VALUE"].map(normalize_id).eq(target_transformer_id),
        "ID",
    ]
)

if not linked_end_ids:
    raise ValueError(f"No transformer ends reference transformer {target_transformer_id}.")

transformer_end_properties = end_properties[
    end_properties["ID"].isin(linked_end_ids)
]

def all_values(properties, property_name):
    values = properties.loc[properties["KEY"].eq(property_name), "VALUE"]
    return "; ".join(sorted({str(value) for value in values}))

winding_rows = []
for end_id, properties in transformer_end_properties.groupby("ID"):
    winding_rows.append(
        {
            "transformer_end_id": end_id,
            "name": all_values(properties, "IdentifiedObject.name"),
            "end_number": all_values(properties, "TransformerEnd.endNumber"),
            "rated_voltage_kV": all_values(properties, "PowerTransformerEnd.ratedU"),
            "rated_apparent_power_MVA": all_values(properties, "PowerTransformerEnd.ratedS"),
        }
    )

windings = pd.DataFrame(winding_rows).sort_values(
    ["end_number", "transformer_end_id"]
).reset_index(drop=True)
display(windings)

print(f"Transformer: NL_TR2_2 ({target_transformer_id})")
print("Nominal winding voltages: " + ", ".join(
    f"{row.end_number}: {row.rated_voltage_kV} kV"
    for row in windings.itertuples()
))

,transformer_end_id,name,end_number,rated_voltage_kV,rated_apparent_power_MVA
0,0dbed103-fc51-4df4-a6fa-0dee4c57f3a3,NL_TR2_2; NL_TR2_3; NL_TR2_4,1,220,1260
1,41ca9e70-1cb4-4971-b8e6-a97a15580b89,NL_TR2_2,2,15.75,1260


Transformer: NL_TR2_2 (2184f365-8cd5-4b5d-8a28-9d68603bb6a4)
Nominal winding voltages: 1: 220 kV, 2: 15.75 kV


The results of this task also exposes a mistake in the model, the first winding of the transformer NL_TR2_2 has a duplicate-ID issue, which would not be allowed as RDF identifiers and CGMES mRID values must uniquely identify one object. Reusing them means the XML contains multiple distinct objects with the same identity.

### Task 3.3.
We investigate the permanently and temporarily allowed limit for line segment NL-Line_5 (ID: _e8acf6b6-99cb-45ad-b8dc-16c7866a4ddc)

In [10]:
target_line_id = "e8acf6b6-99cb-45ad-b8dc-16c7866a4ddc"
target_line_name = "NL-Line_5"

# Normalize references because triplets removes the RDF '#' fragment and
# the original rdf:ID uses a leading underscore.
def normalize_id(value):
    return str(value).lstrip("#_")

line_properties = triples[triples["ID"].map(normalize_id).eq(target_line_id)]
if line_properties.empty:
    raise ValueError(f"Line {target_line_name} ({target_line_id}) was not found.")

line_name = line_properties.loc[
    line_properties["KEY"].eq("IdentifiedObject.name"),
    "VALUE",
].iloc[0]

# Find the terminals whose conducting equipment is NL-Line_5.
line_terminal_ids = set(
    triples.loc[
        triples["KEY"].eq("Terminal.ConductingEquipment")
        & triples["VALUE"].map(normalize_id).eq(target_line_id),
        "ID",
    ]
)

# Map each operational-limit set to its terminal.
limit_set_terminal_rows = triples[
    triples["KEY"].eq("OperationalLimitSet.Terminal")
    & triples["ID"].isin(
        set(
            triples.loc[
                triples["KEY"].eq("OperationalLimitSet.Terminal")
                & triples["VALUE"].isin(line_terminal_ids),
                "ID",
            ]
        )
    )
]
line_limit_set_ids = set(limit_set_terminal_rows["ID"])
limit_set_to_terminal = dict(
    zip(limit_set_terminal_rows["ID"], limit_set_terminal_rows["VALUE"])
)

# Find operational limits belonging to those limit sets.
line_limit_ids = set(
    triples.loc[
        triples["KEY"].eq("OperationalLimit.OperationalLimitSet")
        & triples["VALUE"].isin(line_limit_set_ids),
        "ID",
    ]
)
limit_properties = triples[triples["ID"].isin(line_limit_ids)]

limit_type_ids = set(
    limit_properties.loc[
        limit_properties["KEY"].eq("OperationalLimit.OperationalLimitType"),
        "VALUE",
    ]
)
limit_type_properties = triples[triples["ID"].isin(limit_type_ids)]

kind_by_type = {}
for limit_type_id, properties in limit_type_properties.groupby("ID"):
    kind = properties.loc[
        properties["KEY"].eq("OperationalLimitType.kind"),
        "VALUE",
    ]
    duration = properties.loc[
        properties["KEY"].eq("OperationalLimitType.acceptableDuration"),
        "VALUE",
    ]
    kind_by_type[limit_type_id] = {
        "limit_kind": kind.iloc[0] if not kind.empty else None,
        "acceptable_duration_s": duration.iloc[0] if not duration.empty else None,
    }

limit_rows = []
for limit_id, properties in limit_properties.groupby("ID"):
    limit_type_id = properties.loc[
        properties["KEY"].eq("OperationalLimit.OperationalLimitType"),
        "VALUE",
    ].iloc[0]
    limit_set_id = properties.loc[
        properties["KEY"].eq("OperationalLimit.OperationalLimitSet"),
        "VALUE",
    ].iloc[0]
    value = float(
        properties.loc[
            properties["KEY"].eq("CurrentLimit.normalValue"),
            "VALUE",
        ].iloc[0]
    )
    type_data = kind_by_type[limit_type_id]
    limit_rows.append(
        {
            "terminal_id": limit_set_to_terminal[limit_set_id],
            "limit_kind": type_data["limit_kind"],
            "acceptable_duration_s": type_data["acceptable_duration_s"],
            "current_limit_A": value,
        }
    )

limits = pd.DataFrame(limit_rows).sort_values(
    ["terminal_id", "limit_kind"]
).reset_index(drop=True)
display(limits)

permanent_limits = limits.loc[
    limits["limit_kind"].eq("LimitKind.patl"),
    "current_limit_A",
]
temporary_limits = limits.loc[
    limits["limit_kind"].eq("LimitKind.tatl"),
    "current_limit_A",
]

if permanent_limits.empty or temporary_limits.empty:
    raise ValueError("Both permanent and temporary limits were not found.")

permanent_limit_a = permanent_limits.iloc[0]
temporary_limit_a = temporary_limits.iloc[0]
print(f"Line: {line_name} ({target_line_id})")
print(f"Permanent allowed limit (PATL): {permanent_limit_a:.0f} A")
print(f"Temporary allowed limit (TATL): {temporary_limit_a:.0f} A")
print(f"Temporary limit duration: 600 s")
print(f"Difference, permanent - temporary: {permanent_limit_a - temporary_limit_a:.0f} A")

,terminal_id,limit_kind,acceptable_duration_s,current_limit_A
0,757d4f50-707b-47a0-891c-cbaefd649631,LimitKind.patl,None,1876.0
1,757d4f50-707b-47a0-891c-cbaefd649631,LimitKind.tatl,600,500.0
2,ae588863-b154-451d-978a-7ab08ac50fb6,LimitKind.patl,None,1876.0
3,ae588863-b154-451d-978a-7ab08ac50fb6,LimitKind.tatl,600,500.0


Line: NL-Line_5 (e8acf6b6-99cb-45ad-b8dc-16c7866a4ddc)
Permanent allowed limit (PATL): 1876 A
Temporary allowed limit (TATL): 500 A
Temporary limit duration: 600 s
Difference, permanent - temporary: 1376 A


The permanently allowed limit (PATL) is 1876A. and temporarily allowed limit (TATL) is 500A for this line segment. 

This again seems like an error in the model, as the TATL should be higher than the PATL. 
500 A TATL vs 1876 A PATL doesn't make physical sense. A 10-minute overload rating (TATL) should always be higher or eaqual to the continuous rating (PATL), often noticeably higher (commonly 1.1–1.5× or more, depending on the TSO's thermal rating methodology). Having TATL at roughly a quarter of PATL is not a legitimate operating limit relationship. By the logic in this model the equipment is only allowed to carry more current continuously than it's allowed to carry briefly under overload, which is backwards.

This is likely a model error due to the fact that every single TATL value is set to 500 A regardless of the associated equipment's actual current rating. Additionally, the per unit resistance values of the ACLineSegment.r for NL-Line_5 seem consistant with a bundeled high voltage power line.

### Task 3.4.
We investigate which generator is set as slack in the given model. 

Power grid models typically designate one generator as the slack bus, which serves as the reference point for voltage in the system. More specifically, the slack bus establishes a voltage angle reference (usually set to 0 degrees), which a solver needs in order to converge on a single solution during power flow analysis. The slack generator also compensates for any imbalance between generation and load, both active and reactive power, ensuring that the system remains balanced.

A physcial analogy to this would be a connection point to a large, stiff grid, which can absorb fluctuations in power without significant changes in voltage or frequency. In practice, the slack generator is often a large conventional power plant, such as gas power plant, which has the capability to adjust its output quickly to maintain system stability.

In [11]:
# Inspect generator and control attributes that may indicate a slack candidate.
type_rows = triples[triples["KEY"].eq("Type")]

def normalize_id(value):
    return str(value).lstrip("#_")

def property_map(object_id):
    rows = triples[triples["ID"].eq(object_id)]
    return dict(zip(rows["KEY"], rows["VALUE"]))

def first_property(properties, *names):
    for name in names:
        if name in properties:
            return properties[name]
    return None

machine_ids = set(
    type_rows.loc[
        type_rows["VALUE"].eq("SynchronousMachine"),
        "ID",
    ]
)
machine_rows = []
for machine_id in machine_ids:
    properties = property_map(machine_id)
    control_reference = first_property(
        properties,
        "RegulatingCondEq.RegulatingControl",
    )
    machine_rows.append(
        {
            "machine_id": machine_id,
            "generator_name": properties.get("IdentifiedObject.name"),
            "generating_unit_id": first_property(
                properties,
                "RotatingMachine.GeneratingUnit",
            ),
            "rated_voltage_kV": properties.get("RotatingMachine.ratedU"),
            "rated_power_MVA": properties.get("RotatingMachine.ratedS"),
            "min_reactive_power_Mvar": properties.get("SynchronousMachine.minQ"),
            "max_reactive_power_Mvar": properties.get("SynchronousMachine.maxQ"),
            "regulating_control_id": control_reference,
            "has_voltage_regulating_control": control_reference is not None,
        }
    )

generator_controls = pd.DataFrame(machine_rows)

# Add generating-unit names and control details to the evidence table.
generating_unit_name_by_id = {}
for unit_id in set(generator_controls["generating_unit_id"].dropna()):
    generating_unit_name_by_id[unit_id] = property_map(unit_id).get(
        "IdentifiedObject.name"
    )
generator_controls["generating_unit_name"] = generator_controls[
    "generating_unit_id"
].map(generating_unit_name_by_id)

def control_details(control_id):
    if not control_id:
        return {
            "control_name": None,
            "control_mode": None,
            "control_terminal": None,
        }
    properties = property_map(control_id)
    return {
        "control_name": properties.get("IdentifiedObject.name"),
        "control_mode": properties.get("RegulatingControl.mode"),
        "control_terminal": properties.get("RegulatingControl.Terminal"),
    }

control_data = generator_controls["regulating_control_id"].map(control_details)
control_data = pd.DataFrame(control_data.tolist(), index=generator_controls.index)
generator_controls = pd.concat([generator_controls, control_data], axis=1)
generator_controls = generator_controls.sort_values("generator_name").reset_index(drop=True)
display(generator_controls)

voltage_regulated = generator_controls[
    generator_controls["has_voltage_regulating_control"]
]
print("Generators with explicit voltage-regulating control:")
print(", ".join(voltage_regulated["generator_name"].dropna()) or "None")
print()
print(
    "Interpretation: the EQ profile identifies a voltage-regulated generator "
    "as a possible slack candidate, but it does not conclusively encode the "
    "load-flow slack/reference designation."
)

,machine_id,generator_name,generating_unit_id,rated_voltage_kV,...,generating_unit_name,control_name,control_mode,control_terminal
0,9c3b8f97-7972-477d-9dc8-87365cc0ad0e,NL-G1,049438a6-780a-44fe-a788-ebe385d98e25,15.75,...,Gen-12923,NL-G1,RegulatingControlModeKind.voltage,faab7959-f9bf-421b-bc3f-d364e0c1388b
1,2844585c-0d35-488d-a449-685bcd57afbf,NL-G2,ca80ee09-3bed-4884-bc28-6dc89d067289,15.75,...,Gen-12910,None,None,None
2,1dc9afba-23b5-41a0-8540-b479ed8baf4b,NL-G3,b850063d-eae7-4675-bc98-4642d3076783,None,...,Gen-12908,None,None,None


Generators with explicit voltage-regulating control:
NL-G1

Interpretation: the EQ profile identifies a voltage-regulated generator as a possible slack candidate, but it does not conclusively encode the load-flow slack/reference designation.
